In [55]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [62]:
class RoPEAttention(nn.Module):
    
    def __init__(self, embed_size, block_size):
        super().__init__()
        self.embed_size = embed_size
        self.key = nn.Linear(embed_size, embed_size)
        self.query = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)

        thetas = 10000 ** -((torch.arange(embed_size) // 2) * 2 / embed_size)
        angles = torch.arange(block_size, dtype = torch.float)[:, None] @ thetas[None, :] # (block_size, embed_size)
        self.register_buffer('cos', torch.cos(angles))
        self.register_buffer('sin', torch.sin(angles))

    def rotate(self, x):
        cosx = x * self.cos[:x.shape[-2]] # (batch_size, seq_len, embed_size) + (seq_len, embed_size)
        swappedx = torch.stack((-x[..., 1::2], x[..., ::2]), dim = -1).flatten(-2)
        sinx = swappedx * self.sin[:x.shape[-2]]
        return cosx + sinx

    def forward(self, x, mask): # mask - (seq_len, seq_len), 1 - visible, 0 - invisible
        # x - (batch_size, seq_len, embed_size)
        K = self.key(x) # (batch_size, seq_len, embed_size)
        Q = self.query(x) # (batch_size, seq_len, embed_size)
        V = self.value(x) # (batch_size, seq_len, embed_size)

        K = self.rotate(K)
        Q = self.rotate(Q)

        Att = Q @ K.transpose(-1, -2) # (batch_size, seq_len, embed_size) @ (batch_size, embed_size, seq_len) -> (batch_size, seq_len, seq_len)

        Att = Att /  (self.embed_size ** 0.5)
        maskedAtt = torch.masked_fill(Att, mask == 0, float('-inf'))
        normAtt = F.softmax(maskedAtt, dim = -1)

        return normAtt @ V # (batch_size, block_size, block_size) @ (batch_size, block_size, embed_size) -> (batch_size, block_size, embed_size)

In [63]:
x = torch.randn((32, 64, 10))

In [66]:
model = RoPEAttention(10, 64)

In [68]:
model(x, torch.ones(64, 64)).shape

torch.Size([32, 64, 10])